In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_2.py ──
"""
Shared infrastructure for MLFP03 Exercise 2 — Regularisation and
Cross-Validation.

Contains: data loading for the Singapore credit scoring dataset,
feature preparation, synthetic 1D bias-variance problem, alpha grids,
plotting helpers, and a shared OUTPUT_DIR for generated artefacts.

Technique-specific code (Ridge/Lasso model construction, nested CV
loops, learning curves) lives in the per-technique solution files —
this module holds only the helpers those files share.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

# Deterministic seed for every random operation in this exercise
SEED = 42

# Alpha sweep used by Ridge / Lasso / ElasticNet and the regularisation path
ALPHAS: list[float] = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# Output directory for HTML plots, summary CSVs, etc.
OUTPUT_DIR = Path("outputs") / "mlfp03_ex2_regularisation_cv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring
# ════════════════════════════════════════════════════════════════════════


def load_credit_data() -> (
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]
):
    """Load + preprocess the MLFP02 Singapore credit scoring parquet.

    Returns:
        X_train, y_train, X_test, y_test, feature_names

    Target: predicts ``credit_utilization`` (continuous ratio). Falls
    back to ``income_sgd`` if the utilisation column is absent in older
    dataset revisions.

    Uses ``kailash_ml.PreprocessingPipeline`` for normalisation +
    ordinal encoding + median imputation. All regularised models
    REQUIRE normalised features (otherwise the penalty is unevenly
    distributed across the coefficient vector).
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    target_col = (
        "credit_utilization" if "credit_utilization" in credit.columns else "income_sgd"
    )

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=credit,
        target=target_col,
        train_size=0.8,
        seed=SEED,
        normalize=True,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != target_col]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=target_col,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=target_col,
    )

    return X_train, y_train, X_test, y_test, col_info["feature_columns"]


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 1D PROBLEM — for bias/variance and polynomial fits
# ════════════════════════════════════════════════════════════════════════


def make_sine_dataset(
    n: int = 100,
    noise_sigma: float = 0.2,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """Generate a 1D noisy-sine regression problem.

    Returns:
        x_train_2d (n_train, 1), y_train (n_train,),
        x_test_2d (n_test, 1),   y_test  (n_test,),
        noise_variance (float, σ² — the irreducible error floor)

    The true function is ``y = sin(2πx) + ε`` with ε ~ N(0, σ²).
    The noise variance is returned so callers can use it in the
    bias-variance decomposition (σ² is the "irreducible noise" term).
    """
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    y = np.sin(2 * np.pi * x) + rng.normal(0, noise_sigma, n)

    split = int(n * 0.8)
    x_train = x[:split].reshape(-1, 1)
    x_test = x[split:].reshape(-1, 1)
    y_train = y[:split]
    y_test = y[split:]
    return x_train, y_train, x_test, y_test, noise_sigma**2


def make_poly_pipeline(degree: int) -> Pipeline:
    """Polynomial-features + scaler + linear-regression pipeline."""
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree, include_bias=False)),
            ("scaler", StandardScaler()),
            ("lr", LinearRegression()),
        ]
    )


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_header(title: str) -> None:
    """Print a banner so each phase of a technique file is easy to spot."""
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)


def format_results_table(rows: list[dict[str, Any]], cols: list[str]) -> str:
    """Render a small table of dicts as fixed-width text."""
    header = "  ".join(f"{c:>12}" for c in cols)
    sep = "-" * len(header)
    body = "\n".join(
        "  ".join(
            f"{row[c]:>12.4f}" if isinstance(row[c], float) else f"{row[c]:>12}"
            for c in cols
        )
        for row in rows
    )
    return f"{header}\n{sep}\n{body}"


def save_html_plot(fig: Any, name: str) -> Path:
    """Write a plotly figure into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / name
    fig.write_html(str(path))
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 2.1: Bias-Variance Trade-off
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Diagnose underfitting vs overfitting from train/test error gaps
#   - Decompose expected test error into Bias², Variance, and irreducible
#     noise via bootstrap resampling
#   - Read a bias-variance curve and pick the "sweet spot" complexity
#   - Connect the bias-variance picture to Singapore credit-risk decisions
#
# PREREQUISITES:
#   - MLFP03 Exercise 1 (feature engineering, sklearn basics)
#   - MLFP02 Module 2 (linear regression, expectation & variance)
#
# ESTIMATED TIME: ~35 minutes
#
# TASKS (5-phase R10):
#   1. Theory — why "more complex" is not always "better"
#   2. Build — polynomial pipelines at increasing degrees
#   3. Train — fit each degree, collect train/test MSE
#   4. Visualise — bias² / variance / noise curve via bootstrap
#   5. Apply — DBS Singapore consumer credit scoring risk
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.metrics import mean_squared_error



## THEORY — The Bias-Variance Decomposition


In [ ]:
#   E[(y - ŷ)²]  =  Bias²(ŷ)  +  Var(ŷ)  +  σ²
#
# HIGH BIAS = "even on average we get the wrong answer" (underfit).
# HIGH VARIANCE = "our answer depends wildly on the training sample"
# (overfit). The irreducible σ² is the floor that no model can beat.



## TASK 2 — BUILD the polynomial experiment


In [ ]:
print_header("Polynomial Degree Experiment on sin(2πx)")

# TODO: Generate the synthetic 1D dataset. Use make_sine_dataset with
# n=100, noise_sigma=0.2, seed=SEED. It returns a tuple of five values:
# x_train (shape n_train,1), y_train, x_test, y_test, noise_variance.
x_train, y_train, x_test, y_test, noise_variance = ____

print(
    f"Train: {x_train.shape[0]} pts  "
    f"Test: {x_test.shape[0]} pts  "
    f"σ² = {noise_variance:.4f}"
)



## TASK 3 — TRAIN polynomial models at many degrees


In [ ]:
degree_rows: dict[int, dict[str, float]] = {}
print(f"\n{'Degree':>6} {'Train MSE':>12} {'Test MSE':>12} {'Gap':>10}  Diagnosis")
print("-" * 60)
for degree in [1, 2, 4, 6, 9, 12, 15, 20]:
    # TODO: Construct the pipeline via make_poly_pipeline(degree), fit
    # it on (x_train, y_train), and compute train/test MSE using
    # sklearn.metrics.mean_squared_error.
    model = ____
    model.fit(____, ____)
    train_mse = ____
    test_mse = ____
    gap = test_mse - train_mse

    # Hint: classify by the `degree` heuristic in the solution comments.
    if degree <= 2:
        diagnosis = "underfit (bias)"
    elif degree <= 6:
        diagnosis = "good fit"
    else:
        diagnosis = "overfit (variance)"

    degree_rows[degree] = {
        "train_mse": train_mse,
        "test_mse": test_mse,
        "gap": gap,
    }
    print(f"{degree:>6} {train_mse:>12.4f} {test_mse:>12.4f} {gap:>10.4f}  {diagnosis}")



### Checkpoint 1


In [ ]:
assert (
    degree_rows[1]["test_mse"] > degree_rows[4]["test_mse"]
), "Degree=1 should have higher test error than degree=4 (underfit)"
assert (
    degree_rows[20]["train_mse"] < degree_rows[2]["train_mse"]
), "Degree=20 should memorise training data (lowest train MSE)"
print("\n[ok] Checkpoint 1 passed — underfit/overfit pattern confirmed")



## TASK 4 — VISUALISE the bias-variance decomposition


Estimate Bias², Variance, and total expected error for a polynomial.


In [ ]:
rng = np.random.default_rng(SEED)


def bias_variance_decomposition(degree: int, n_bootstrap: int = 50) -> dict[str, float]:
    all_preds = []
    for _ in range(n_bootstrap):
        # TODO: Draw a bootstrap index with rng.choice(len(y_train),
        # len(y_train), replace=True), fit a polynomial pipeline on the
        # resampled data, and append the test-set predictions.
        idx = ____
        model = make_poly_pipeline(degree)
        model.fit(____, ____)
        all_preds.append(model.predict(x_test))

    preds = np.array(all_preds)
    mean_pred = preds.mean(axis=0)

    # Noiseless truth at the test points
    y_truth = np.sin(2 * np.pi * x_test.ravel())

    # TODO: Compute the three components of expected test error.
    # bias_sq  = mean of (mean_pred - y_truth)^2
    # variance = mean of preds.var(axis=0)
    bias_sq = ____
    variance = ____
    expected = bias_sq + variance + noise_variance

    return {
        "bias_sq": float(bias_sq),
        "variance": float(variance),
        "noise": noise_variance,
        "expected_error": float(expected),
    }


print_header("Bias-Variance Decomposition via Bootstrap")
print(
    f"{'Degree':>6} {'Bias²':>10} {'Variance':>10} {'Noise':>8} "
    f"{'Expected':>12}  Dominant"
)
print("-" * 60)

bv_rows: dict[int, dict[str, float]] = {}
for degree in [1, 2, 3, 6, 10, 15]:
    bv = bias_variance_decomposition(degree, n_bootstrap=40)
    bv_rows[degree] = bv
    dominant = "Bias" if bv["bias_sq"] > bv["variance"] else "Variance"
    print(
        f"{degree:>6} {bv['bias_sq']:>10.4f} {bv['variance']:>10.4f} "
        f"{bv['noise']:>8.4f} {bv['expected_error']:>12.4f}  {dominant}"
    )



### Checkpoint 2


In [ ]:
assert (
    bv_rows[1]["bias_sq"] > bv_rows[10]["bias_sq"]
), "Degree=1 should have higher bias² than degree=10"
assert (
    bv_rows[1]["variance"] < bv_rows[15]["variance"]
), "Degree=1 should have lower variance than degree=15"
print("\n[ok] Checkpoint 2 passed — bias-variance decomposition valid")



## TASK 5 — APPLY: DBS Singapore Consumer Credit Scoring


Stakeholder         | Concern                      | BV trade-off
--------------------|------------------------------|-----------------
Credit risk team    | Missed defaults              | Too much bias
Model-risk / MAS    | Out-of-time instability      | Too much variance
Branch / CX         | Denied good customers        | Too much bias
Finance / capital   | Buffer volatility            | Too much variance


In [ ]:
# SCENARIO: DBS Bank (Singapore) scores ~2M active retail credit-card
# customers every night. A 5-feature linear scorecard is interpretable
# but high-bias; a 180-feature unregularised GBT is high-variance and
# fails MAS out-of-time validation.
#
# BUSINESS IMPACT: Every 0.10 pp reduction in default rate from a
# well-regularised model is worth ~S$18M/year on the S$18B retail book.
# MAS Notice 637 capital treatment rewards stable models with ~3% of
# the book (~S$540M) freed regulatory capital.

print_header("DBS Retail Credit Scoring — Bias/Variance in Context")
print(
)



## REFLECTION


======================================================================
  WHAT YOU'VE MASTERED
======================================================================

  [x] Train/test error gap as an overfit diagnostic
  [x] The E[(y-ŷ)²] = Bias² + Variance + σ² decomposition
  [x] Empirical bias/variance estimation via bootstrap
  [x] Applying the trade-off to a real Singapore credit decision

  NEXT: 02_ridge_regression.py — shrink coefficients toward zero with
  L2 and meet its Bayesian alter-ego (Gaussian prior on weights).


In [ ]:
print(
)

